In [14]:
import sys
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.getcwd())

from src.splits import create_lesion_splits
from src.dataset import HAM10000Dataset, get_train_transforms, get_eval_transforms, compute_class_weights
import pandas as pd
import numpy as np
import torch

In [2]:
df = pd.read_csv('./data/HAM10000_metadata.csv')
df.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [3]:
train_df, test_df, val_df = create_lesion_splits(df, random_state=10, val_size=0.15, test_size=0.15, tolerance=0.03)

✓ No lesion_id overlap across train/val/test
  Train: 5228 lesions, Val: 1121 lesions, Test: 1121 lesions
✓ train split class proportions within 3% of full dataset (max deviation: 0.000)
✓ val split class proportions within 3% of full dataset (max deviation: 0.000)
✓ test split class proportions within 3% of full dataset (max deviation: 0.000)


In [4]:
image_dir = os.path.join(os.getcwd(), 'data', 'HAM10000_images')
train_dataset = HAM10000Dataset(train_df, image_dir, None)

In [5]:
print(train_dataset.__len__)
print(train_dataset.__getitem__(4))

<bound method HAM10000Dataset.__len__ of <src.dataset.HAM10000Dataset object at 0x000002029E019F90>>
(<PIL.Image.Image image mode=RGB size=600x450 at 0x202AA30D990>, 2)


In [ ]:
train_dataset = HAM10000Dataset(train_df, image_dir, get_train_transforms())
print(train_dataset.__len__)
img, label = train_dataset.__getitem__(4)
assert img.shape == (3, 224, 224), f"Unexpected shape: {img.shape}"
assert isinstance(label, (int, np.integer)), f"Expected int, got {type(label)}"
assert 0 <= label <= 6, f"Label out of range: {label}"

<bound method HAM10000Dataset.__len__ of <src.dataset.HAM10000Dataset object at 0x00000202AB03FC90>>


In [11]:
for i in range(len(train_dataset)):
    try:
        img, label = train_dataset[i]
    except Exception as e:
        print(f"Failed at index {i}: {e}")

In [12]:
val_dataset = HAM10000Dataset(val_df, image_dir, get_eval_transforms())
img1, _ = val_dataset[0]
img2, _ = val_dataset[0]
assert torch.equal(img1, img2), "Eval transform is not deterministic!"

In [13]:
img1, _ = train_dataset[0]
img2, _ = train_dataset[0]
assert not torch.equal(img1, img2), "Train transform doesn't appear to be augmenting!"

In [ ]:
weights = compute_class_weights(train_df)
print(weights)


tensor([ 4.5877,  2.7678,  1.3031, 12.5589,  1.2702,  0.2133, 10.2522])
